In [ ]:
import xml.etree.ElementTree as ET
import json
import re

# Paths
xml_path = 'My EndNote Library.xml'
output_path = 'articles_with_abstracts.json'

# Compile DOI regex
doi_pattern = re.compile(r'(10\.\d{4,9}/[^\s"\'<]+)')

# Parse XML
tree = ET.parse(xml_path)
root = tree.getroot()

articles = []

for record in root.findall('.//record'):
    # Title
    title_elem = record.find('.//titles/title/style')
    title = title_elem.text.strip() if title_elem is not None else ''
    
    # First author surname
    author_elem = record.find('.//contributors/authors/author/style')
    author_text = author_elem.text.strip() if author_elem is not None else ''
    author = author_text.split(',')[0]
    
    # Abstract
    abstract_elem = record.find('.//abstract/style')
    abstract = abstract_elem.text.strip() if abstract_elem is not None else ''
    if not abstract:
        continue
    
    # DOI from <electronic-resource-num>
    doi_elem = record.find('.//electronic-resource-num')
    doi = doi_elem.text.strip() if doi_elem is not None else ''
    
    # Fallback: regex search in record XML string
    if not doi or not doi_pattern.match(doi):
        record_str = ET.tostring(record, encoding='unicode')
        m = doi_pattern.search(record_str)
        doi = m.group(1) if m else None
    
    articles.append({
        "title": title,
        "author": author,
        "doi": doi,
        "abstract": abstract
    })

# Write JSON
with open(output_path, 'w') as f:
    json.dump(articles, f, indent=2)

print(f"Generated {output_path}")
